## 2.6 Evaluation

The multimodal RAG assistant was evaluated using 10 test questions covering:
- Document-based construction safety questions
- Image-based questions
- Multimodal questions combining document requirements with YOLO observations
- Questions where the provided safety document does not contain enough information

The evaluation checks:
- Retrieval relevance
- Groundedness of the generated answer
- Correct use of YOLO visual observations
- Hallucination or unsupported claims
- Failure cases and mitigation strategies

### Evaluation Results

| # | Question | RAG Relevant | YOLO Used | Result | Failure Case / Mitigation |
|---|---|---|---|---|---|
| 1 | When should workers wear hard hats? | Yes | Yes | Pass | — |
| 2 | What safety equipment is visible in this image? | Not required | Yes | Pass | PDF source was unnecessary for an image-only question |
| 3 | What protection should workers use for their hands? | Yes | Yes | Pass | Initial response did not mention detections; prompt was strengthened |
| 4 | What protection should workers use for their feet? | Yes | Yes | Pass | — |
| 5 | What protection should workers use for their eyes and face? | Yes | Yes | Pass | — |
| 6 | When should workers use hearing protection? | Yes | Yes | Pass | — |
| 7 | What are the main situations where head protection is required? | Yes | Yes | Pass | — |
| 8 | What does the safety document require for head protection, and what does the image show about head protection? | Yes | Yes | Pass after mitigation | Initial response inferred non-compliance from NO-Hardhat detections; prompt was strengthened to prohibit compliance inference |
| 9 | What type of respiratory protection does the safety document require? | Insufficient | Yes | Partial | The answer avoided hallucinating respiratory requirements but added unrelated hearing-protection information |
| 10 | What does YOLO detect about head protection in this image? | Not required | Yes | Partial | The answer unnecessarily referenced the document instead of directly answering from YOLO observations |

### Failure Cases and Mitigation

The evaluation identified several failure cases:

**Failure Case 1 — Missing image observations**

In an early test, the model answered the document-based question correctly but failed to mention the YOLO detections even though detections were available.

**Mitigation:**  
The generation prompt was updated to require explicit reporting of relevant computer vision observations whenever detections are provided.

---

**Failure Case 2 — Inferring safety compliance from YOLO**

For a multimodal head-protection question, the model initially interpreted `NO-Hardhat` detections as evidence of non-compliance.

**Mitigation:**  
The prompt was strengthened to explicitly state that YOLO detections are visual observations only and must never be used to label a worker as safe, unsafe, compliant, or non-compliant.

---

**Failure Case 3 — Irrelevant retrieved information**

For a question about respiratory protection, the provided document did not contain enough information. The model correctly avoided inventing a respiratory requirement, but it added unrelated hearing-protection information.

**Mitigation:**  
The generation prompt should instruct the model to answer only information relevant to the user's question and avoid unrelated safety categories.

---

**Failure Case 4 — Image-focused question incorrectly referring to the document**

For a question specifically asking what YOLO detects, the model unnecessarily stated that the document did not provide YOLO information.

**Mitigation:**  
For image-focused questions, the generation prompt should prioritize the computer vision observations and treat YOLO as the source of image detections rather than the safety document.

### Evaluation Summary

The evaluation demonstrates that the system can combine retrieved construction safety information with YOLO visual observations while maintaining source grounding.

The main observed weaknesses were:
1. Irrelevant information included in some answers.
2. Initial attempts to infer compliance from visual detections.
3. Unnecessary references to the document for image-only questions.

These issues were addressed through prompt-level mitigation, especially by explicitly separating document requirements from computer vision observations.

In [1]:
import pandas as pd

evaluation_data = [
    {
        "Test": 1,
        "Question": "When should workers wear hard hats?",
        "RAG Relevant": "Yes",
        "YOLO Used": "Yes",
        "Grounded": "Yes",
        "Result": "Pass",
        "Failure": "None"
    },
    {
        "Test": 2,
        "Question": "What safety equipment is visible in this image?",
        "RAG Relevant": "Not required",
        "YOLO Used": "Yes",
        "Grounded": "Yes",
        "Result": "Pass",
        "Failure": "PDF source was unnecessary for an image-only question"
    },
    {
        "Test": 3,
        "Question": "What protection should workers use for their hands?",
        "RAG Relevant": "Yes",
        "YOLO Used": "Yes",
        "Grounded": "Yes",
        "Result": "Pass",
        "Failure": "Initial response did not mention detections; prompt was strengthened"
    },
    {
        "Test": 4,
        "Question": "What protection should workers use for their feet?",
        "RAG Relevant": "Yes",
        "YOLO Used": "Yes",
        "Grounded": "Yes",
        "Result": "Pass",
        "Failure": "None"
    },
    {
        "Test": 5,
        "Question": "What protection should workers use for their eyes and face?",
        "RAG Relevant": "Yes",
        "YOLO Used": "Yes",
        "Grounded": "Yes",
        "Result": "Pass",
        "Failure": "None"
    },
    {
        "Test": 6,
        "Question": "When should workers use hearing protection?",
        "RAG Relevant": "Yes",
        "YOLO Used": "Yes",
        "Grounded": "Yes",
        "Result": "Pass",
        "Failure": "None"
    },
    {
        "Test": 7,
        "Question": "What are the main situations where head protection is required?",
        "RAG Relevant": "Yes",
        "YOLO Used": "Yes",
        "Grounded": "Yes",
        "Result": "Pass",
        "Failure": "None"
    },
    {
        "Test": 8,
        "Question": "What does the safety document require for head protection, and what does the image show about head protection?",
        "RAG Relevant": "Yes",
        "YOLO Used": "Yes",
        "Grounded": "Yes",
        "Result": "Pass after mitigation",
        "Failure": "Initial response inferred non-compliance from NO-Hardhat detections"
    },
    {
        "Test": 9,
        "Question": "What type of respiratory protection does the safety document require?",
        "RAG Relevant": "Insufficient",
        "YOLO Used": "Yes",
        "Grounded": "Partial",
        "Result": "Partial",
        "Failure": "Unrelated hearing-protection information was added"
    },
    {
        "Test": 10,
        "Question": "What does YOLO detect about head protection in this image?",
        "RAG Relevant": "Not required",
        "YOLO Used": "Yes",
        "Grounded": "Partial",
        "Result": "Partial",
        "Failure": "Model unnecessarily referenced the document"
    }
]

evaluation_df = pd.DataFrame(evaluation_data)

display(evaluation_df)

,Test,Question,RAG Relevant,YOLO Used,Grounded,Result,Failure
0,1,When should workers wear hard hats?,Yes,Yes,Yes,Pass,None
1,2,What safety equipment is visible in this image?,Not required,Yes,Yes,Pass,PDF source was unnecessary for an image-only q...
2,3,What protection should workers use for their h...,Yes,Yes,Yes,Pass,Initial response did not mention detections; p...
3,4,What protection should workers use for their f...,Yes,Yes,Yes,Pass,None
4,5,What protection should workers use for their e...,Yes,Yes,Yes,Pass,None
5,6,When should workers use hearing protection?,Yes,Yes,Yes,Pass,None
6,7,What are the main situations where head protec...,Yes,Yes,Yes,Pass,None
7,8,What does the safety document require for head...,Yes,Yes,Yes,Pass after mitigation,Initial response inferred non-compliance from ...
8,9,What type of respiratory protection does the s...,Insufficient,Yes,Partial,Partial,Unrelated hearing-protection information was a...
9,10,What does YOLO detect about head protection in...,Not required,Yes,Partial,Partial,Model unnecessarily referenced the document


In [2]:
total_tests = len(evaluation_df)

passed_tests = evaluation_df["Result"].str.startswith("Pass").sum()

partial_tests = (evaluation_df["Result"] == "Partial").sum()

pass_rate = (passed_tests / total_tests) * 100

print(f"Total tests: {total_tests}")
print(f"Passed tests: {passed_tests}")
print(f"Partial tests: {partial_tests}")
print(f"Pass rate: {pass_rate:.1f}%")

Total tests: 10
Passed tests: 8
Partial tests: 2
Pass rate: 80.0%


## 2.7 Export

The trained components required by the RAG pipeline are exported for use by the backend application.

The exported artifacts include:
- Chroma persistent vector store
- Embedding model configuration
- Collection name
- Retrieval configuration
- YOLO model path
- Confidence threshold
- Ollama model configuration

The backend loads these components during application startup instead of rebuilding the pipeline for every request.

In [3]:
from pathlib import Path
import os

# Project paths
project_dir = Path.cwd().parent
vector_store_path = project_dir / "chroma_db"

# Exported artifacts
print("Exported RAG artifacts:")
print(f"Vector store: {vector_store_path}")
print(f"Exists: {vector_store_path.exists()}")

if vector_store_path.exists():
    print("Vector store contents:")
    for item in vector_store_path.iterdir():
        print(" -", item.name)

# Configuration used by the backend
export_config = {
    "embedding_model": "all-MiniLM-L6-v2",
    "chroma_collection": "construction_safety",
    "top_k": 2,
    "ollama_model": "llama3.2",
    "yolo_confidence_threshold": 0.30,
}

print("\nPipeline configuration:")
for key, value in export_config.items():
    print(f"{key}: {value}")

Exported RAG artifacts:
Vector store: d:\ai2 ITI\Final Project\Construction-Hazard-Detection.v97i.yolov11\chroma_db
Exists: True
Vector store contents:
 - chroma.sqlite3
 - e084fe57-f3f3-4374-bdda-8c7b95957f10

Pipeline configuration:
embedding_model: all-MiniLM-L6-v2
chroma_collection: construction_safety
top_k: 2
ollama_model: llama3.2
yolo_confidence_threshold: 0.3
